# Chapter 9 — Memory: Types, Retrieval and Hybrid Stores

*Memory is not chat history.*

## Objective

**Why.** An agent needs more than chat history. It must answer different questions about what it knows — *what do we know about X?*, *how is X related to Y?*, *what just happened?* — and they are not the same question.

**What.** Three access patterns, because the questions differ: recency, similarity (vector), and relationships (graph) — plus a hybrid that blends all three.

**How.** Build `ShortTermMemory`, `VectorMemory`, `GraphMemory`, and `HybridMemory` over the six-policy corpus and compare what each retrieves for the same query.

In [ ]:
from pathlib import Path

from glassloop.memory import (
    GraphMemory, HybridMemory, MemoryItem, MemoryKind,
    ShortTermMemory, Triple, VectorMemory, chunk_text,
)

policies_dir = Path('data/policies')
if not policies_dir.exists():
    policies_dir = Path('..') / 'data' / 'policies'
print(f'corpus: {sorted(p.name for p in policies_dir.glob("*.txt"))}')

## The embedder

`VectorMemory` takes any object with an `embed(texts)` method that returns one vector per text. We use the GMS semantic encoder (`all-MiniLM-L6-v2`, loaded locally), so retrieval is on real meaning, the same model serves the GMS store later in the chapter, and results stay reproducible (a fixed local model is deterministic).

In [ ]:
from knowlytix.core.graph.encoders import encode_texts

class GMSEmbedder:
    """Real semantic embeddings from the GMS stack (all-MiniLM-L6-v2, local)."""
    dim = 384

    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        self.model_name = model_name

    def embed(self, texts):
        return encode_texts(list(texts), model_name=self.model_name).tolist()

## VectorMemory: chunk, embed, retrieve

We load every policy file, chunk it and add the chunks. `query` returns the top-k by cosine similarity.

In [ ]:
vec = VectorMemory(GMSEmbedder())
for path in sorted(policies_dir.glob('*.txt')):
    text = path.read_text()
    for i, chunk in enumerate(chunk_text(text, window=200, overlap=20)):
        vec.add(MemoryItem(
            content=chunk,
            kind=MemoryKind.POLICY,
            metadata={'source': path.name, 'chunk': i},
        ))
print(f'loaded {len(vec)} chunks')

hits = vec.query('overdraft fee policy', k=3)
for h in hits:
    print(f'  [{h.metadata["source"]}#{h.metadata["chunk"]}] {h.content[:80]!r}')

## GraphMemory: triples and multi-hop

Useful when retrieval has to follow relations — `policy A defines process B that references requirement C`.

In [ ]:
g = GraphMemory()
g.add_triple(Triple('overdraft_policy', 'governs',  'overdraft_fee'))
g.add_triple(Triple('overdraft_fee',    'reversible_via', 'fee_reversal_policy'))
g.add_triple(Triple('fee_reversal_policy', 'requires', 'manager_approval'))

two_hops = g.neighbors('overdraft_policy', hops=2)
for t in two_hops:
    print(f'  {t.subject} --[{t.relation}]--> {t.object}')

## ShortTermMemory: bounded recency

Useful for the active task: the last few user messages, the last few tool results. Substring matching, not semantic.

In [ ]:
short = ShortTermMemory(maxlen=4)
for msg in ['hi', 'I was charged a fee', 'specifically an overdraft fee', 'help me dispute it']:
    short.add(MemoryItem(content=msg, kind=MemoryKind.SHORT_TERM))
print(short.query('fee'))

## Hybrid: combine the three

`HybridMemory` adds an item to all three backends and scores `α · sim + β · graph + γ · recency`. The weights are constructor args; defaults favor vector similarity.

In [ ]:
hybrid = HybridMemory(
    vector=VectorMemory(GMSEmbedder()),
    graph=GraphMemory(),
    short_term=ShortTermMemory(maxlen=8),
)
for path in sorted(policies_dir.glob('*.txt')):
    hybrid.add(MemoryItem(content=path.read_text()[:200], kind=MemoryKind.POLICY, metadata={'source': path.name}))

results = hybrid.query('overdraft fee', k=3)
for h in results:
    print(f'  [{h.metadata.get("source", "?")}] {h.content[:80]!r}')

## An optional fourth tier: exact numbers + contradiction rejection (GMS)

Vector, graph and recency leave two gaps: **byte-exact numeric recall** (an LLM in the path eventually misreads a number) and **contradiction-on-write** (a vector store silently accepts facts that conflict with its own contents). When correctness is critical — a fee, a threshold, an approval limit — the GMS substrate closes both: a trained geometric triple store, queried through the `GMSMemory` adapter, adds Exact Numerical Memory (`lookup_enm`) and contradiction scoring (`score_triple`).

This tier is **optional** — skip it if vector + graph + recency cover your needs. The trained store ships in `data/gms_banking_store/`; `load()` returns it with no retraining. The geometry and threshold calibration are deferred to the consolidated GMS chapter and Appendix C.

In [ ]:
import torch
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from glassloop.gms_backend import GMSMemory

_root = Path('.') if Path('data/gms_banking_store').exists() else Path('..')
store = GMSExpertStore(
    DocGMSConfig(store_path=str(_root / 'data' / 'gms_banking_store')),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
store.load()
mem = GMSMemory(store)

# Win 1 — Exact Numerical Memory: values are returned losslessly, never parsed.
for cat, eid in [('fee_schedule',       'overdraft/per_occurrence'),
                 ('fee_schedule',       'wire_international/per_transaction'),
                 ('reversal_authority', 'supervisor')]:
    print(f'  {cat}/{eid} = {mem.lookup_enm(cat, eid)}')

# Win 2 — contradiction rejection on a functional relation (one tail per head).
for head, tail, label in [('representative', '35.0',  'committed'),
                          ('representative', '100.0', 'contradicts')]:
    print(f'  ({head}, has_max_reversal, {tail}) -> '
          f'{mem.score_triple(head, "has_max_reversal", tail):.3f}  {label}')

## Anti-patterns flagged here

- Treating chat history as memory.
- Top-k retrieval without reranking when the corpus is heterogeneous.
- Embedding without a chunking strategy.

In [ ]:
# Self-check: at least one retrieval hit references an overdraft policy.
assert any('overdraft' in h.content.lower() or 'overdraft' in str(h.metadata).lower() for h in results)
print('OK')

## Exercises

Worked solutions follow. Same six-policy corpus; these compound toward the capstone's grounded, provenance-backed responses. Exercise 9.3 uses the small open-weight model (Qwen2.5-3B-Instruct) — first load takes ~30s.

In [ ]:
# Exercise 9.1 — does chunk size change what survives retrieval?
for window, overlap in [(120, 0), (200, 20), (400, 40)]:
    v = VectorMemory(GMSEmbedder())
    for path in sorted(policies_dir.glob('*.txt')):
        for i, chunk in enumerate(chunk_text(path.read_text(), window=window, overlap=overlap)):
            v.add(MemoryItem(content=chunk, kind=MemoryKind.POLICY,
                             metadata={'source': path.name, 'chunk': i}))
    top = [h.metadata['source'] for h in v.query('overdraft fee policy', k=3)]
    print(f'  window={window:<3} overlap={overlap:<2} -> {top}')
# With a real semantic embedder the top hit (overdraft.txt) is stable across all
# three windows: each policy is short and single-topic, so chunk boundaries don't
# change the match. Chunking starts to matter on long docs (a fact split across a
# boundary) or large heterogeneous corpora (many near-duplicate chunks compete).

In [ ]:
# Exercise 9.2 — a provenance chain the audit log can replay
g2 = GraphMemory()
for s, r, o in [('customer_anna', 'disputes', 'charge_8841'),
                ('charge_8841', 'governed_by', 'overdraft_policy'),
                ('overdraft_policy', 'reversible_via', 'fee_reversal_policy'),
                ('fee_reversal_policy', 'requires', 'manager_approval')]:
    g2.add_triple(Triple(s, r, o))

def support_chain(graph, start):
    """Follow FORWARD edges only (neighbors returns inbound edges too)."""
    chain, cur, seen = [], start, {start}
    while True:
        nxt = [t for t in graph.neighbors(cur, hops=1)
               if t.subject == cur and t.object not in seen]
        if not nxt:
            break
        t = nxt[0]; chain.append(t); cur = t.object; seen.add(cur)
    return chain

for t in support_chain(g2, 'charge_8841'):
    print(f'  {t.subject} -[{t.relation}]-> {t.object}')

In [ ]:
# Exercise 9.3 — small-model extraction, GMS-verified number
import re, json
from transformers import AutoModelForCausalLM, AutoTokenizer

_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
_lm = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", dtype=torch.bfloat16, device_map="cuda")
text = (policies_dir / 'overdraft.txt').read_text()
_sys = ("Extract factual (subject, relation, object) triples from the policy text. "
        "Return ONLY a JSON list of [subject, relation, object] string arrays.")
_ids = _tok.apply_chat_template(
    [{"role": "system", "content": _sys}, {"role": "user", "content": text}],
    add_generation_prompt=True, return_tensors="pt").to(_lm.device)
_out = _lm.generate(_ids, max_new_tokens=300, do_sample=False,
                    pad_token_id=_tok.eos_token_id)
_gen = _tok.decode(_out[0, _ids.shape[1]:], skip_special_tokens=True)
triples = json.loads(re.search(r'\[.*\]', _gen, re.DOTALL).group(0))

kg = GraphMemory()
for tr in triples:
    if isinstance(tr, list) and len(tr) == 3:
        kg.add_triple(Triple(*map(str, tr)))
print(f'extracted {len(triples)} triples; sample: {triples[:3]}')

# Small model is fine for STRUCTURE; the NUMBER must come from Exact Numerical Memory.
nums = [float(n) for tr in triples for cell in tr
        for n in re.findall(r'\$?\s*(\d+(?:\.\d+)?)', str(cell))]
exact = mem.lookup_enm('fee_schedule', 'overdraft/per_occurrence')
model_fee = next((n for n in nums if n == exact), nums[0] if nums else None)
print(f'model numbers={nums} | ENM exact={exact} | '
      f'{"MATCH" if model_fee == exact else "MISMATCH - flag for review"}')